<a href="https://colab.research.google.com/github/ropaniagnes72-jpg/KinesisIQ-affordable-Housing-in-Early_Waring/blob/main/ingestion_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Pod 3 — Kinesis: City of Toronto Open Data Ingestion Pipeline
Extracts, cleans, geocodes, and loads Toronto housing datasets into SQLite
(populating both the standardized 13-column `events` table and `kinesis_events` / `fact_*` tables).

Usage:
    python ingest_toronto.py --db kinesis.db
"""

import argparse
import hashlib
import json
import os
import re
import sqlite3
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import gpd
    import geopandas as gpd
    HAS_GPD = True
except ImportError:
    HAS_GPD = False

try:
    from pyproj import Transformer
    HAS_PYPROJ = True
except ImportError:
    HAS_PYPROJ = False


# ==============================================================================
# 1. Configuration & Constants
# ==============================================================================
CKAN = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
AS_OF = pd.Timestamp.today().normalize()
FORECAST_YEARS = 5

RS_RED_BELOW = 70
RS_DROP_POINTS = 5
RS_AUDIT_PERCENTILE = 0.025

PIPELINE_ASSUMPTIONS = {
    "completed":          (0.00, 1, 1),
    "under_construction": (0.95, 1, 3),
    "approved":           (0.70, 2, 5),
    "planning":           (0.40, 3, 7),
    "other":              (0.40, 3, 7),
}

WARDS = [
    (1, "Etobicoke North"), (2, "Etobicoke Centre"), (3, "Etobicoke-Lakeshore"),
    (4, "Parkdale-High Park"), (5, "York South-Weston"), (6, "York Centre"),
    (7, "Humber River-Black Creek"), (8, "Eglinton-Lawrence"), (9, "Davenport"),
    (10, "Spadina-Fort York"), (11, "University-Rosedale"), (12, "Toronto-St. Paul's"),
    (13, "Toronto Centre"), (14, "Toronto-Danforth"), (15, "Don Valley West"),
    (16, "Don Valley East"), (17, "Don Valley North"), (18, "Willowdale"),
    (19, "Beaches-East York"), (20, "Scarborough Southwest"), (21, "Scarborough Centre"),
    (22, "Scarborough-Agincourt"), (23, "Scarborough North"), (24, "Scarborough-Guildwood"),
    (25, "Scarborough-Rouge Park"),
]
WARD_NAME = dict(WARDS)


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


WARD_NAME_TO_ID = {_norm(n): i for i, n in WARDS}

KINESIS_EVENT_COLS = [
    "event_id", "event_type", "event_date", "event_year", "event_quarter",
    "ward_id", "neighbourhood_id", "address", "units_affected",
    "units_affordable", "source_dataset", "data_quality_flag",
    "units_lost", "geo_method",
]

STANDARD_EVENT_COLS = [
    "event_id", "source", "anon_id", "site_id", "zone", "event_time",
    "modality", "text_input", "payload", "consent_method",
    "consent_scope", "privacy_token", "geo_json",
]

GEO_FLAG = {
    "spatial_join": "clean",
    "source_ward_field": "clean",
    "ward_grid_lookup": "ward_grid_inferred",
    "unmapped": "unmapped_ward",
}


# ==============================================================================
# 2. CKAN & Parsing Helpers
# ==============================================================================
def ckan_get(action: str, **params) -> dict:
    r = requests.get(f"{CKAN}/api/3/action/{action}", params=params, timeout=90)
    r.raise_for_status()
    j = r.json()
    if not j.get("success"):
        raise RuntimeError(f"CKAN {action} failed: {j}")
    return j["result"]


def find_package(slug: str, *search_terms: str) -> dict:
    try:
        return ckan_get("package_show", id=slug)
    except Exception:
        pass
    for term in search_terms:
        hits = ckan_get("package_search", q=term, rows=10)["results"]
        if hits:
            return hits[0]
    raise LookupError(f"No CKAN package found for '{slug}' / {search_terms}")


def _base_name(name: str) -> str:
    n = str(name or "").lower().strip()
    n = re.sub(r"\.(csv|xlsx|xls|json|geojson)$", "", n)
    n = re.sub(r"\s*[-_]\s*(4326|2952|wgs84|mtm\w*)\s*$", "", n)
    return n.strip()


def pick_resources(pkg: dict, name_filter: Optional[str] = None) -> Tuple[list, list]:
    cands = []
    for r in pkg["resources"]:
        if name_filter and not re.search(name_filter, r.get("name", ""), re.I):
            continue
        fmt = (r.get("format") or "").upper()
        if r.get("datastore_active"):
            src = f"{CKAN}/datastore/dump/{r['id']}"
        elif fmt == "CSV":
            src = r["url"]
        else:
            continue
        projected = 1 if re.search(r"2952|mtm", r.get("name", "") + " " + r.get("url", ""), re.I) else 0
        cands.append((_base_name(r.get("name")), projected, 0 if r.get("datastore_active") else 1, r, src))

    chosen, skipped = {}, []
    for base, proj, notds, r, src in sorted(cands, key=lambda t: (t[0], t[1], t[2])):
        if base in chosen:
            skipped.append(r.get("name"))
        else:
            chosen[base] = (r, src)
    return list(chosen.values()), skipped


def coalesce_columns(df: pd.DataFrame) -> pd.DataFrame:
    groups = {}
    for c in df.columns:
        if not str(c).startswith("_"):
            key = re.sub(r"[^a-z0-9]+", "", str(c).lower())
            groups.setdefault(key, []).append(c)
    for cols in groups.values():
        if len(cols) > 1:
            first = cols[0]
            for c in cols[1:]:
                df[first] = df[first].combine_first(df[c])
            df = df.drop(columns=cols[1:])
    return df


def load_package_tables(slug: str, *search_terms: str, dedupe: bool = True) -> pd.DataFrame:
    pkg = find_package(slug, *search_terms)
    chosen, _ = pick_resources(pkg)
    frames = []
    for i, (r, src) in enumerate(chosen):
        try:
            df = pd.read_csv(src, low_memory=False)
        except Exception as e:
            print(f"  [WARN] Skipped resource '{r.get('name')}': {e}")
            continue
        ids = df["_id"] if "_id" in df.columns else df.index
        df["_uid"] = [f"{i}-{v}" for v in ids]
        df["_resource"] = r.get("name")
        frames.append(df)

    if not frames:
        raise RuntimeError(f"No loadable tabular resource in package '{pkg['name']}'")

    out = coalesce_columns(pd.concat(frames, ignore_index=True))
    if dedupe:
        content = [c for c in out.columns if c not in ("_id", "_uid", "_resource")]
        out = out.drop_duplicates(subset=content, keep="first").reset_index(drop=True)
    return out


def pick_col(df: pd.DataFrame, *patterns: str) -> Optional[str]:
    for p in patterns:
        for c in df.columns:
            name = str(c)
            if name.startswith("_"):
                continue
            if (re.search(p, name, re.I)
                    or re.search(p, re.sub(r"[^A-Za-z0-9]+", "_", name), re.I)
                    or re.search(p, re.sub(r"[^A-Za-z0-9]+", " ", name), re.I)):
                return c
    return None


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.replace(r"[,$\s]", "", regex=True), errors="coerce")


def to_dt(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s) and s.dropna().size and s.dropna().median() > 1e11:
        return pd.to_datetime(s, unit="ms", errors="coerce")
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors="coerce")


def sane_dates(d: pd.Series, lo: str = "1990-01-01", hi_years_ahead: int = 1) -> pd.Series:
    hi = AS_OF + pd.DateOffset(years=hi_years_ahead)
    return d.where((d >= pd.Timestamp(lo)) & (d <= hi))


def sha256_token(val: str) -> str:
    return hashlib.sha256(str(val).encode("utf-8")).hexdigest()[:32]


def to_utc_iso(d: pd.Series) -> pd.Series:
    """Localize naive timestamps to America/Toronto and convert to ISO-8601 UTC."""
    localized = d.dt.tz_localize("America/Toronto", ambiguous="NaT", nonexistent="shift_forward")
    return localized.dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")


# ==============================================================================
# 3. Spatial & Geography Helpers
# ==============================================================================
def ward_from_value(v) -> Optional[int]:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip()
    m = re.fullmatch(r"(?:ward\s*)?0*(\d{1,2})(?:\.0+)?", s, re.I) or re.search(r"\(\s*0*(\d{1,2})\s*\)", s)
    if m and 1 <= int(m.group(1)) <= 25:
        return int(m.group(1))
    return WARD_NAME_TO_ID.get(_norm(re.sub(r"\(.*?\)", "", s)))


def load_geo(slug: str, *search_terms: str):
    pkg = find_package(slug, *search_terms)
    res = [r for r in pkg["resources"] if (r.get("format") or "").upper() == "GEOJSON"]
    pref = [r for r in res if "4326" in r.get("name", "") or "4326" in r.get("url", "")]
    r = (pref or res)[0]
    g = gpd.read_file(r["url"])
    if g.crs is None:
        g = g.set_crs(4326)
    return g.to_crs(4326)


def load_reference_geography():
    wards_gdf, nbhd_gdf = None, None
    nbhd_table = pd.DataFrame(columns=["neighbourhood_id", "neighbourhood_name"])
    if not HAS_GPD:
        return wards_gdf, nbhd_gdf, nbhd_table

    g_w = load_geo("city-wards", "city wards")
    w_col = pick_col(g_w, r"^AREA_SHORT_CODE$", r"^AREA_LONG_CODE$", r"WARD.*(NUM|ID|CODE)")
    g_w["ward_id"] = pd.to_numeric(g_w[w_col].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    g_w = g_w[g_w["ward_id"].between(1, 25)].copy()
    g_w["ward_id"] = g_w["ward_id"].astype(int)
    wards_gdf = g_w[["ward_id", "geometry"]].copy()

    g_n = load_geo("neighbourhoods", "neighbourhoods")
    n_col = pick_col(g_n, r"^AREA_SHORT_CODE$", r"^AREA_LONG_CODE$", r"NEIGHBOURHOOD.*(ID|NUM|CODE)")
    n_name = pick_col(g_n, r"^AREA_NAME$", r"NEIGHBOURHOOD.*NAME")
    g_n["neighbourhood_id"] = pd.to_numeric(g_n[n_col].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    g_n = g_n.dropna(subset=["neighbourhood_id"]).copy()
    g_n["neighbourhood_id"] = g_n["neighbourhood_id"].astype(int)
    g_n["neighbourhood_name"] = g_n[n_name].astype(str).str.replace(r"\s*\(\d+\)\s*$", "", regex=True)
    nbhd_gdf = g_n[["neighbourhood_id", "neighbourhood_name", "geometry"]].copy()
    nbhd_table = nbhd_gdf[["neighbourhood_id", "neighbourhood_name"]].drop_duplicates()

    return wards_gdf, nbhd_gdf, nbhd_table


def lonlat_from_geometry(s: pd.Series) -> Tuple[pd.Series, pd.Series]:
    lon = np.full(len(s), np.nan)
    lat = np.full(len(s), np.nan)
    for i, v in enumerate(s.to_numpy(dtype=object)):
        if isinstance(v, str) and v[:1] == "{":
            try:
                g = json.loads(v)
            except ValueError:
                continue
            c, t = g.get("coordinates"), g.get("type")
            if t == "MultiPoint" and c:
                c = c[0]
            if t in ("Point", "MultiPoint") and c and len(c) >= 2:
                lon[i], lat[i] = c[0], c[1]
    return pd.Series(lat, index=s.index), pd.Series(lon, index=s.index)


def lonlat_from_xy(x: pd.Series, y: pd.Series) -> Tuple[pd.Series, pd.Series]:
    x, y = to_num(x), to_num(y)
    nan = pd.Series(np.nan, index=x.index)
    if not HAS_PYPROJ:
        return nan, nan
    ok = x.notna() & y.notna() & (x > 0) & (y > 0)
    if not ok.any():
        return nan, nan

    samp = ok[ok].index[:5000]
    best = None
    for crs in ["EPSG:2952", "EPSG:2019", "EPSG:26917"]:
        tr = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
        lo, la = tr.transform(x[samp].to_numpy(), y[samp].to_numpy())
        share = float(((np.asarray(la) > 43.4) & (np.asarray(la) < 44.0)
                       & (np.asarray(lo) > -79.8) & (np.asarray(lo) < -78.9)).mean())
        if best is None or share > best[0]:
            best = (share, crs)

    if best[0] < 0.95:
        return nan, nan
    tr = Transformer.from_crs(best[1], "EPSG:4326", always_xy=True)
    lo, la = tr.transform(x[ok].to_numpy(), y[ok].to_numpy())
    lat, lon = nan.copy(), nan.copy()
    lat[ok], lon[ok] = la, lo
    return lat, lon


def add_coords(df: pd.DataFrame, lat_col=None, lon_col=None, geom_col=None, x_col=None, y_col=None) -> pd.DataFrame:
    lat = pd.Series(np.nan, index=df.index)
    lon = lat.copy()

    def fill(la, lo):
        nonlocal lat, lon
        good = la.between(43.4, 44.0) & lo.between(-79.8, -78.9) & lat.isna()
        if good.any():
            lat = lat.where(~good, la)
            lon = lon.where(~good, lo)

    if lat_col and lon_col:
        fill(to_num(df[lat_col]), to_num(df[lon_col]))
    if geom_col and geom_col in df.columns:
        fill(*lonlat_from_geometry(df[geom_col]))
    if x_col and y_col and x_col in df.columns and y_col in df.columns and lat.isna().any():
        fill(*lonlat_from_xy(df[x_col], df[y_col]))
    return df.assign(_lat=lat, _lon=lon)


def finalize_geo(df: pd.DataFrame, wards_gdf, nbhd_gdf, ward_col=None, lat_col=None, lon_col=None) -> pd.DataFrame:
    out = df.copy()
    out["ward_id_sp"] = np.nan
    out["neighbourhood_id_sp"] = np.nan

    if HAS_GPD and wards_gdf is not None and lat_col in out and lon_col in out:
        lat = pd.to_numeric(out[lat_col], errors="coerce")
        lon = pd.to_numeric(out[lon_col], errors="coerce")
        ok = lat.between(43.4, 44.0) & lon.between(-79.8, -78.9)
        if ok.any():
            pts = gpd.GeoDataFrame(
                {"_i": out.index[ok]},
                geometry=gpd.points_from_xy(lon[ok], lat[ok]),
                crs="EPSG:4326",
            )
            for gdf, idcol, outcol in ((wards_gdf, "ward_id", "ward_id_sp"),
                                       (nbhd_gdf, "neighbourhood_id", "neighbourhood_id_sp")):
                if gdf is not None:
                    j = gpd.sjoin(pts, gdf[[idcol, "geometry"]], how="left", predicate="within")
                    j = j[~j.index.duplicated(keep="first")]
                    out.loc[j["_i"].values, outcol] = j[idcol].values

    src = pd.to_numeric(out[ward_col].map(ward_from_value), errors="coerce") if ward_col else pd.Series(np.nan, index=out.index)
    ward = out["ward_id_sp"].where(out["ward_id_sp"].notna(), src)
    out["geo_method"] = np.where(
        out["ward_id_sp"].notna(), "spatial_join",
        np.where(src.notna(), "source_ward_field", "unmapped")
    )
    out["ward_id"] = pd.to_numeric(ward, errors="coerce").astype("Int64")
    out["neighbourhood_id"] = pd.to_numeric(out["neighbourhood_id_sp"], errors="coerce").astype("Int64")
    return out.drop(columns=["ward_id_sp", "neighbourhood_id_sp"])


# ==============================================================================
# 4. Database Schema & Idempotent Writers
# ==============================================================================
def init_db(db_path: str, nbhd_table: pd.DataFrame) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS dim_ward (
                ward_id INTEGER PRIMARY KEY,
                ward_name VARCHAR(100) NOT NULL,
                ward_code VARCHAR(20) UNIQUE NOT NULL
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS dim_neighbourhood (
                neighbourhood_id INTEGER PRIMARY KEY,
                neighbourhood_name VARCHAR(150) NOT NULL
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS kinesis_events (
                event_id VARCHAR(60) PRIMARY KEY,
                event_type VARCHAR(50) NOT NULL,
                event_date DATE NOT NULL,
                event_year INTEGER NOT NULL,
                event_quarter INTEGER,
                ward_id INTEGER REFERENCES dim_ward(ward_id),
                neighbourhood_id INTEGER REFERENCES dim_neighbourhood(neighbourhood_id),
                address TEXT,
                units_affected INTEGER DEFAULT 0,
                units_affordable INTEGER DEFAULT 0,
                source_dataset VARCHAR(100) NOT NULL,
                data_quality_flag VARCHAR(30) DEFAULT 'clean',
                units_lost INTEGER DEFAULT 0,
                geo_method VARCHAR(30)
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS events (
                event_id TEXT PRIMARY KEY,
                source TEXT NOT NULL,
                anon_id TEXT NOT NULL,
                site_id TEXT NOT NULL,
                zone TEXT,
                event_time TEXT NOT NULL,
                modality TEXT NOT NULL,
                text_input TEXT,
                payload TEXT,
                consent_method TEXT NOT NULL,
                consent_scope TEXT NOT NULL,
                privacy_token TEXT NOT NULL,
                geo_json TEXT,
                ingested_at TEXT DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now'))
            )
        """)
        cur.executemany(
            "INSERT OR REPLACE INTO dim_ward VALUES (?,?,?)",
            [(i, n, f"WARD_{i:02d}") for i, n in WARDS],
        )
        if len(nbhd_table):
            cur.executemany(
                "INSERT OR REPLACE INTO dim_neighbourhood VALUES (?,?)",
                [(int(r.neighbourhood_id), r.neighbourhood_name) for r in nbhd_table.itertuples()],
            )
        conn.commit()


def write_events_idempotent(
    db_path: str,
    kinesis_df: pd.DataFrame,
    standard_df: pd.DataFrame,
    source_name: str,
) -> None:
    with sqlite3.connect(db_path) as conn:
        conn.execute("DELETE FROM kinesis_events WHERE source_dataset = ?", (source_name,))
        k_rows = kinesis_df[KINESIS_EVENT_COLS].astype(object).where(kinesis_df.notna(), None).values.tolist()
        conn.executemany(
            f"INSERT OR REPLACE INTO kinesis_events ({','.join(KINESIS_EVENT_COLS)}) "
            f"VALUES ({','.join('?' * len(KINESIS_EVENT_COLS))})",
            k_rows,
        )

        conn.execute("DELETE FROM events WHERE source = ?", (source_name,))
        s_rows = standard_df[STANDARD_EVENT_COLS].astype(object).where(standard_df.notna(), None).values.tolist()
        conn.executemany(
            f"INSERT OR REPLACE INTO events ({','.join(STANDARD_EVENT_COLS)}) "
            f"VALUES ({','.join('?' * len(STANDARD_EVENT_COLS))})",
            s_rows,
        )
        conn.commit()


def write_table(db_path: str, df: pd.DataFrame, name: str) -> None:
    with sqlite3.connect(db_path) as conn:
        df.to_sql(name, conn, if_exists="replace", index=False)


# ==============================================================================
# 5. Dataset Ingestion Pipelines
# ==============================================================================
def ingest_demolition_replacement(db_path: str, wards_gdf, nbhd_gdf) -> int:
    source_name = "Demolition and Replacement of Rental Housing Units"
    raw = load_package_tables(
        "demolition-and-replacement-of-rental-housing-units",
        "demolition and replacement of rental housing units",
    )
    demo = raw.copy()
    c = {k: pick_col(demo, *p) for k, p in {
        "dem_tot": [r"^Total Rental Homes for Demolition"],
        "dem_aff": [r"^Affordable Rental Homes for Demolition"],
        "dem_mid": [r"^Mid-Range Rental Homes for Demolition"],
        "dem_high": [r"^High-End Rental Homes for Demolition"],
        "rep_tot": [r"^Total Rental Homes Replaced"],
        "rep_aff": [r"^Affordable Rental Homes Replaced"],
        "rep_mid": [r"^Mid-Range Rental Homes Replaced"],
        "rep_high": [r"^High-End Rental Homes Replaced"],
        "permit_dt": [r"^Date the Rental Demolition Permit"],
        "council_dt": [r"Council Approval Date"],
        "address": [r"^Address of Existing Rental Building"],
        "ward": [r"Post 2018.*Ward", r"Ward"],
    }.items()}

    for k in ["dem_tot", "dem_aff", "dem_mid", "dem_high", "rep_tot", "rep_aff", "rep_mid", "rep_high"]:
        demo[k] = to_num(demo[c[k]]).fillna(0) if c[k] else 0

    permit_dt = sane_dates(to_dt(demo[c["permit_dt"]]))
    council_dt = sane_dates(to_dt(demo[c["council_dt"]]))
    demo["event_date"] = permit_dt.fillna(council_dt)
    demo["date_basis"] = np.where(
        permit_dt.notna(), "permit_issued",
        np.where(council_dt.notna(), "council_approval", "none")
    )
    demo = finalize_geo(demo, wards_gdf, nbhd_gdf, ward_col=c["ward"])
    demo["net_affordable_change"] = demo["rep_aff"] - demo["dem_aff"]
    demo["affordable_replacement_ratio"] = np.where(demo["dem_aff"] > 0, demo["rep_aff"] / demo["dem_aff"], np.nan)

    fact_demo = demo[[
        "_uid", c["address"], "ward_id", "geo_method", "event_date", "date_basis",
        "dem_tot", "dem_aff", "dem_mid", "dem_high", "rep_tot", "rep_aff", "rep_mid", "rep_high",
        "net_affordable_change", "affordable_replacement_ratio",
    ]].rename(columns={c["address"]: "address"})
    write_table(db_path, fact_demo, "fact_demolition_replacement")

    keep = demo["event_date"].notna()
    d = demo[keep].copy()

    k_ev = pd.DataFrame({
        "event_id": "DEMO_" + d["_uid"].astype(str),
        "event_type": "RENTAL_DEMOLITION",
        "event_date": d["event_date"].dt.strftime("%Y-%m-%d"),
        "event_year": d["event_date"].dt.year,
        "event_quarter": d["event_date"].dt.quarter,
        "ward_id": d["ward_id"],
        "neighbourhood_id": d["neighbourhood_id"],
        "address": d[c["address"]].astype(str).str.strip(),
        "units_affected": d["dem_tot"].astype(int),
        "units_affordable": d["dem_aff"].astype(int),
        "source_dataset": source_name,
        "data_quality_flag": d["geo_method"].map(GEO_FLAG),
        "units_lost": 0,
        "geo_method": d["geo_method"],
    })

    s_ev = pd.DataFrame({
        "event_id": "DEMO_" + d["_uid"].astype(str),
        "source": source_name,
        "anon_id": [sha256_token(f"{a}|{u}") for a, u in zip(d[c["address"]], d["_uid"])],
        "site_id": [f"WARD_{int(w):02d}" if pd.notna(w) else "UNMAPPED" for w in d["ward_id"]],
        "zone": d[c["address"]].astype(str).str.strip(),
        "event_time": to_utc_iso(d["event_date"]),
        "modality": "RENTAL_DEMOLITION",
        "text_input": d["date_basis"],
        "payload": [
            json.dumps({
                "dem_tot": int(r.dem_tot),
                "dem_aff": int(r.dem_aff),
                "dem_mid": int(r.dem_mid),
                "dem_high": int(r.dem_high),
                "rep_tot": int(r.rep_tot),
                "rep_aff": int(r.rep_aff),
                "rep_mid": int(r.rep_mid),
                "rep_high": int(r.rep_high),
                "net_affordable_change": int(r.net_affordable_change),
            })
            for r in d.itertuples()
        ],
        "consent_method": "public_data",
        "consent_scope": json.dumps(["analytics", "research"]),
        "privacy_token": "public",
        "geo_json": [
            json.dumps({
                "ward_id": int(w) if pd.notna(w) else None,
                "ward_name": WARD_NAME.get(int(w)) if pd.notna(w) else None,
                "address": str(addr).strip(),
                "geo_method": gm,
            })
            for w, addr, gm in zip(d["ward_id"], d[c["address"]], d["geo_method"])
        ],
    })

    write_events_idempotent(db_path, k_ev, s_ev, source_name)
    return len(s_ev)


def ingest_dev_applications(db_path: str, wards_gdf, nbhd_gdf) -> int:
    source_name = "Development Applications"
    raw = load_package_tables("development-applications", "development applications")
    m = {k: pick_col(raw, *pats) for k, pats in {
        "date": [r"date.?submit", r"submit", r"application.*date", r"^date"],
        "type": [r"application.?type", r"^type"],
        "status": [r"^status"],
        "desc": [r"description"],
        "ward": [r"ward.?(num|id|code)", r"^ward"],
        "lat": [r"^lat"],
        "lon": [r"^lon|^lng"],
        "x": [r"^X$"],
        "y": [r"^Y$"],
        "street_num": [r"street.?num"],
        "street_name": [r"street.?name"],
        "street_type": [r"street.?type"],
        "folder": [r"FOLDERRSN"],
    }.items()}

    dev = raw.copy()
    if m["folder"]:
        dev = dev.drop_duplicates(m["folder"]).reset_index(drop=True)

    dev["event_date"] = sane_dates(to_dt(dev[m["date"]]))
    dev = add_coords(dev, lat_col=m["lat"], lon_col=m["lon"], x_col=m["x"], y_col=m["y"])
    dev = finalize_geo(dev, wards_gdf, nbhd_gdf, ward_col=m["ward"], lat_col="_lat", lon_col="_lon")

    unit_re = re.compile(
        r"(\d[\d,]*)\s*(?:residential|dwelling|rental|affordable|condominium|apartment|new|replacement)?\s*"
        r"(?:residential\s+)?(?:dwelling\s+)?(?:units?|suites?)\b",
        re.I,
    )
    housing_re = re.compile(
        r"residential|dwelling|rental|apartment|condominium|townhouse|mixed[- ]use|housing|\bunits?\b",
        re.I,
    )

    def parse_units(text):
        nums = [int(x.replace(",", "")) for x in unit_re.findall(str(text)) if x.replace(",", "").isdigit()]
        return max(nums) if nums else np.nan

    dev["units_parsed"] = dev[m["desc"]].map(parse_units) if m["desc"] else np.nan
    dev["housing_related"] = (
        (dev[m["desc"]].astype(str).str.contains(housing_re) | dev["units_parsed"].notna())
        if m["desc"] else False
    )

    if m["street_num"] and m["street_name"]:
        n = dev[m["street_num"]].astype(str).str.extract(r"(\d+)")[0]
        st_name = dev[m["street_name"]].fillna("").astype(str).str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
        st_type = (
            dev[m["street_type"]].fillna("").astype(str).str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
            if m["street_type"] else ""
        )
        key = n.fillna("") + "|" + st_name + ("|" + st_type if m["street_type"] else "")
        dev["addr_key"] = key.where(n.notna() & dev[m["street_name"]].notna())
        dev["address"] = (
            dev[m["street_num"]].fillna("").astype(str) + " "
            + dev[m["street_name"]].fillna("").astype(str)
            + (" " + dev[m["street_type"]].fillna("").astype(str) if m["street_type"] else "")
        ).str.strip()
    else:
        dev["addr_key"] = np.nan
        dev["address"] = None

    fact_dev = pd.DataFrame({
        "_uid": dev["_uid"],
        "event_date": dev["event_date"],
        "address": dev["address"],
        "addr_key": dev["addr_key"],
        "app_type": dev[m["type"]] if m["type"] else None,
        "status": dev[m["status"]] if m["status"] else None,
        "units_parsed": dev["units_parsed"],
        "housing_related": dev["housing_related"],
        "ward_id": dev["ward_id"],
        "neighbourhood_id": dev["neighbourhood_id"],
        "geo_method": dev["geo_method"],
    })
    write_table(db_path, fact_dev, "fact_dev_applications")

    keep = dev["event_date"].notna()
    d = dev[keep].copy()

    k_ev = pd.DataFrame({
        "event_id": "DEVAPP_" + d["_uid"].astype(str),
        "event_type": "DEV_APPLICATION",
        "event_date": d["event_date"].dt.strftime("%Y-%m-%d"),
        "event_year": d["event_date"].dt.year,
        "event_quarter": d["event_date"].dt.quarter,
        "ward_id": d["ward_id"],
        "neighbourhood_id": d["neighbourhood_id"],
        "address": d["address"],
        "units_affected": d["units_parsed"].fillna(0).astype(int),
        "units_affordable": 0,
        "source_dataset": source_name,
        "data_quality_flag": d["geo_method"].map(GEO_FLAG),
        "units_lost": 0,
        "geo_method": d["geo_method"],
    })

    folder_series = d[m["folder"]] if m["folder"] else d["_uid"]
    app_type_series = d[m["type"]].fillna("") if m["type"] else pd.Series("", index=d.index)
    status_series = d[m["status"]].fillna("") if m["status"] else pd.Series("", index=d.index)

    s_ev = pd.DataFrame({
        "event_id": "DEVAPP_" + d["_uid"].astype(str),
        "source": source_name,
        "anon_id": [sha256_token(f) for f in folder_series],
        "site_id": [f"WARD_{int(w):02d}" if pd.notna(w) else "UNMAPPED" for w in d["ward_id"]],
        "zone": d["address"],
        "event_time": to_utc_iso(d["event_date"]),
        "modality": "DEV_APPLICATION",
        "text_input": (app_type_series.astype(str) + ": " + status_series.astype(str)).str.strip(": "),
        "payload": [
            json.dumps({
                "units_parsed": int(u) if pd.notna(u) else None,
                "housing_related": bool(hr),
                "app_type": str(at) if pd.notna(at) else None,
                "status": str(st) if pd.notna(st) else None,
                "addr_key": str(ak) if pd.notna(ak) else None,
            })
            for u, hr, at, st, ak in zip(
                d["units_parsed"], d["housing_related"], app_type_series, status_series, d["addr_key"].fillna("")
            )
        ],
        "consent_method": "public_data",
        "consent_scope": json.dumps(["analytics", "research"]),
        "privacy_token": "public",
        "geo_json": [
            json.dumps({
                "ward_id": int(w) if pd.notna(w) else None,
                "neighbourhood_id": int(nb) if pd.notna(nb) else None,
                "lat": round(float(la), 6) if pd.notna(la) else None,
                "lon": round(float(lo), 6) if pd.notna(lo) else None,
                "geo_method": gm,
            })
            for w, nb, la, lo, gm in zip(
                d["ward_id"], d["neighbourhood_id"], d["_lat"], d["_lon"], d["geo_method"]
            )
        ],
    })

    write_events_idempotent(db_path, k_ev, s_ev, source_name)
    return len(s_ev)


def ingest_supporting_facts(db_path: str, wards_gdf, nbhd_gdf) -> None:
    """Ingests non-event snapshot/aggregate tables: Active Stock, Pipeline, and RentSafeTO."""
    # 1. Active Affordable & Social Housing (city-wide quarterly)
    stock_raw = load_package_tables(
        "active-affordable-and-social-housing-units",
        "active affordable and social housing units",
    )
    q_col = pick_col(stock_raw, r"quarter|period")
    aff_col = pick_col(stock_raw, r"afford.*(unit|home)", r"afford")
    sub_col = pick_col(stock_raw, r"subsid.*(unit|home)", r"subsid")

    def parse_quarter(s):
        t = str(s)
        mm = re.search(r"Q([1-4])\D{0,3}(\d{4})", t, re.I)
        if mm:
            q, y = int(mm.group(1)), int(mm.group(2))
        else:
            mm = re.search(r"(\d{4})\D{0,3}Q([1-4])", t, re.I)
            if not mm:
                return pd.NaT
            y, q = int(mm.group(1)), int(mm.group(2))
        return pd.Period(year=y, quarter=q, freq="Q").end_time.normalize()

    fact_stock = pd.DataFrame({
        "quarter_label": stock_raw[q_col].astype(str),
        "quarter_end": stock_raw[q_col].map(parse_quarter),
        "affordable_units": to_num(stock_raw[aff_col]) if aff_col else np.nan,
        "subsidized_units": to_num(stock_raw[sub_col]) if sub_col else np.nan,
    }).sort_values("quarter_end").reset_index(drop=True)
    write_table(db_path, fact_stock, "fact_affordable_stock_city")

    # 2. Affordable Rental Housing Pipeline
    pipe_raw = load_package_tables(
        "upcoming-and-recently-completed-affordable-housing-units",
        "affordable rental housing pipeline",
    )
    mp = {k: pick_col(pipe_raw, *pats) for k, pats in {
        "pid": [r"^Project ID"], "ward": [r"Ward Number", r"^ward"], "geom": [r"^geometry$"],
        "address": [r"^Addresses", r"address"], "units": [r"^Affordable Homes Approved", r"afford.*(unit|home)"],
        "rgi": [r"^RGI"], "supportive": [r"^Supportive"], "status": [r"^status", r"status|stage"],
    }.items()}
    pipe = pipe_raw.drop_duplicates(mp["pid"]).reset_index(drop=True) if mp["pid"] else pipe_raw.copy()
    pipe = add_coords(pipe, geom_col=mp["geom"])
    pipe = finalize_geo(pipe, wards_gdf, nbhd_gdf, ward_col=mp["ward"], lat_col="_lat", lon_col="_lon")

    def pipeline_stage(status):
        t = str(status).lower()
        if re.search(r"occupied|complet", t):
            return "completed"
        if re.search(r"construction|building permit|permit issued", t):
            return "under_construction"
        if re.search(r"approved|council|zoning|opa", t):
            return "approved"
        if re.search(r"application|review|planning|pre-?app|proposal|prelim", t):
            return "planning"
        return "other"

    pipe["units"] = to_num(pipe[mp["units"]]).fillna(0)
    pipe["stage"] = pipe[mp["status"]].map(pipeline_stage)
    pipe["p_deliver"] = pipe["stage"].map(lambda s: PIPELINE_ASSUMPTIONS[s][0])
    pipe["deliver_lo"] = pipe["stage"].map(lambda s: PIPELINE_ASSUMPTIONS[s][1])
    pipe["deliver_hi"] = pipe["stage"].map(lambda s: PIPELINE_ASSUMPTIONS[s][2])
    p_by_y = np.clip((np.minimum(FORECAST_YEARS, pipe["deliver_hi"]) - pipe["deliver_lo"] + 1)
                     / (pipe["deliver_hi"] - pipe["deliver_lo"] + 1), 0, 1)
    pipe[f"expected_units_{FORECAST_YEARS}y"] = pipe["units"] * pipe["p_deliver"] * p_by_y

    fact_pipe = pd.DataFrame({
        "_uid": pipe["_uid"], "project_id": pipe[mp["pid"]], "address": pipe[mp["address"]],
        "status": pipe[mp["status"]], "stage": pipe["stage"], "units": pipe["units"],
        "rgi_units": to_num(pipe[mp["rgi"]]) if mp["rgi"] else np.nan,
        "supportive_units": to_num(pipe[mp["supportive"]]) if mp["supportive"] else np.nan,
        "p_deliver": pipe["p_deliver"], "deliver_lo": pipe["deliver_lo"], "deliver_hi": pipe["deliver_hi"],
        f"expected_units_{FORECAST_YEARS}y": pipe[f"expected_units_{FORECAST_YEARS}y"],
        "ward_id": pipe["ward_id"], "neighbourhood_id": pipe["neighbourhood_id"], "geo_method": pipe["geo_method"],
    })
    write_table(db_path, fact_pipe, "fact_affordable_pipeline")

    # 3. RentSafeTO Apartment Building Evaluation
    rs_raw = load_package_tables("apartment-building-evaluation", "RentSafeTO apartment building evaluation")
    mr = {k: pick_col(rs_raw, *pats) for k, pats in {
        "rsn": [r"^RSN$"], "ward": [r"^WARD$", r"^ward"], "lat": [r"^lat"], "lon": [r"^lon"],
        "address": [r"site.?address", r"address"], "score": [r"CURRENT.*BUILDING.*EVAL.*SCORE", r"EVAL.*SCORE"],
        "units": [r"CONFIRMED.?UNITS", r"unit"], "year_built": [r"YEAR.?BUILT"],
        "eval_date": [r"EVALUATION.?COMPLETED", r"EVAL.*DATE"], "eval_year": [r"YEAR.?EVALUATED"],
    }.items()}
    rs = rs_raw.copy()
    rs["eval_date"] = sane_dates(to_dt(rs[mr["eval_date"]])) if mr["eval_date"] else pd.Series(pd.NaT, index=rs.index)
    if mr["eval_year"]:
        yr_fb = pd.to_datetime(to_num(rs[mr["eval_year"]]).round().astype("Int64").astype(str) + "-07-01", errors="coerce")
        rs["eval_date"] = rs["eval_date"].fillna(yr_fb)
    rs["score_v"] = to_num(rs[mr["score"]]).where(lambda x: x.between(0, 100))
    rs = rs.dropna(subset=["score_v", "eval_date"]).sort_values([mr["rsn"], "eval_date"]).reset_index(drop=True)

    write_table(db_path, rs[[mr["rsn"], "eval_date", "score_v"]].rename(columns={mr["rsn"]: "rsn"}), "fact_rentsafe_history")

    rs["prev_score"] = rs.groupby(mr["rsn"])["score_v"].shift(1)
    rs["prev_date"] = rs.groupby(mr["rsn"])["eval_date"].shift(1)
    rs["n_evals"] = rs.groupby(mr["rsn"]).cumcount() + 1
    latest = rs.groupby(mr["rsn"]).tail(1).copy().reset_index(drop=True)
    latest["score_delta"] = latest["score_v"] - latest["prev_score"]
    latest["days_between_evals"] = (latest["eval_date"] - latest["prev_date"]).dt.days
    latest = finalize_geo(latest, wards_gdf, nbhd_gdf, ward_col=mr["ward"], lat_col=mr["lat"], lon_col=mr["lon"])
    latest["score"] = latest["score_v"]
    latest["units"] = to_num(latest[mr["units"]]).fillna(0) if mr["units"] else 0
    latest["year_built"] = to_num(latest[mr["year_built"]]) if mr["year_built"] else np.nan
    latest["red_band"] = latest["score"] < RS_RED_BELOW
    latest["degrading"] = latest["score_delta"] <= -RS_DROP_POINTS
    latest["at_risk"] = latest["red_band"] | latest["degrading"]
    audit_cut = latest.groupby(latest["eval_date"].dt.year)["score"].transform(lambda x: x.quantile(RS_AUDIT_PERCENTILE))
    latest["audit_priority"] = latest["score"] <= audit_cut

    fact_rs = latest[[
        "_uid", "ward_id", "neighbourhood_id", "geo_method", "score", "prev_score",
        "score_delta", "days_between_evals", "n_evals", "units", "year_built",
        "eval_date", "red_band", "degrading", "at_risk", "audit_priority",
    ]].copy()
    fact_rs["rsn"] = latest[mr["rsn"]]
    fact_rs["address"] = latest[mr["address"]] if mr["address"] else None
    write_table(db_path, fact_rs, "fact_rentsafe")



# 6. Pipeline Entrypoint

def run(db_path: str = "kinesis.db") -> None:
    print(f"[1/4] Loading reference geography (25 wards, 158 neighbourhoods)...")
    wards_gdf, nbhd_gdf, nbhd_table = load_reference_geography()
    init_db(db_path, nbhd_table)

    print(f"[2/4] Ingesting Rental Demolition & Replacement events...")
    n_demo = ingest_demolition_replacement(db_path, wards_gdf, nbhd_gdf)
    print(f"      -> {n_demo:,} RENTAL_DEMOLITION events written.")

    print(f"[3/4] Ingesting Development Applications events...")
    n_dev = ingest_dev_applications(db_path, wards_gdf, nbhd_gdf)
    print()
    print(f"      -> {n_dev:,} DEV_APPLICATION events written._")

    print(f"[4/4] Ingesting supporting fact tables (Active Stock, Pipeline, RentSafeTO)...")
    ingest_supporting_facts(db_path, wards_gdf, nbhd_gdf)
    print(f"Done. Database ready at: {db_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Ingest City of Toronto housing datasets into SQLite.")
    parser.add_argument("--db", default="kinesis.db", help="Path to output SQLite database (default: kinesis.db)")
    # Use parse_known_args to bypass Jupyter/Colab kernel background arguments safely
    args, _ = parser.parse_known_args()
    run(db_path=args.db)


[1/4] Loading reference geography (25 wards, 158 neighbourhoods)...
[2/4] Ingesting Rental Demolition & Replacement events...
      -> 173 RENTAL_DEMOLITION events written.
[3/4] Ingesting Development Applications events...

      -> 8,639 DEV_APPLICATION events written._
[4/4] Ingesting supporting fact tables (Active Stock, Pipeline, RentSafeTO)...
Done. Database ready at: kinesis.db
